# Thermal UAV Detection — AIResQ + HIT-UAV

This notebook fine-tunes one detector with exactly two labels: `person` and `vehicle`. It reuses the extracted AIResQ dataset in Colab and merges HIT-UAV `car`, `bicycle`, and `other vehicle` into `vehicle`. AIResQ currently has person-only annotations, so a partial-label loss preserves those person examples without treating unlabeled vehicles as background. The existing checkpoint initializes training; the new checkpoint is saved separately.


In [ ]:
!pip -q install kaggle ultralytics==8.4.173 pyyaml requests

from pathlib import Path
import json, os, shutil, subprocess, zipfile, yaml, requests, random

WORK_DIR = Path('/content/thermal_person_vehicle')
AIRESQ_ZIP = WORK_DIR / 'AIResQ.zip'
# The prior notebook run extracts AIResQ here. Change this only if you extracted it elsewhere.
AIRESQ_RAW = WORK_DIR / 'airesq_raw'
HIT_DOWNLOAD = WORK_DIR / 'hit_uav_download'
OUTPUT_DIR = WORK_DIR / 'combined_dataset'
WORK_DIR.mkdir(parents=True, exist_ok=True)


## Use the extracted AIResQ files

The notebook reuses `/content/thermal_person_vehicle/airesq_raw` if it already contains the extracted dataset. It only downloads and extracts AIResQ when that folder is absent, so rerunning this cell preserves label edits made in place. The expected extracted layout includes `images/` and `labels/` with YOLO `.txt` files. Keep `labels.json` and `metadata.json` in the source folder if they are present.


In [ ]:
RECORD_ID = '19677785'
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def has_extracted_airesq(root):
    return root.is_dir() and any(
        p.is_dir() and p.name.casefold() in {'images', 'image'}
        and any(f.is_file() and f.suffix.lower() in IMAGE_SUFFIXES for f in p.rglob('*'))
        for p in root.rglob('*')
    ) and any(p.is_dir() and p.name.casefold() in {'labels', 'label'} for p in root.rglob('*'))

if has_extracted_airesq(AIRESQ_RAW):
    print('Reusing extracted AIResQ and its current labels:', AIRESQ_RAW)
else:
    metadata_url = f'https://zenodo.org/api/records/{RECORD_ID}'
    response = requests.get(metadata_url, timeout=60)
    response.raise_for_status()
    record = response.json()
    file_info = next((f for f in record.get('files', []) if f['key'].casefold() == 'airesq.zip'), None)
    if file_info is None:
        raise FileNotFoundError(f'AIResQ.zip is not listed in Zenodo record {RECORD_ID}. Files: {[f["key"] for f in record.get("files", [])]}')

    if not AIRESQ_ZIP.exists() or AIRESQ_ZIP.stat().st_size != file_info['size']:
        partial_zip = AIRESQ_ZIP.with_name(AIRESQ_ZIP.name + '.part')
        download_url = file_info['links'].get('content', file_info['links'].get('self'))
        with requests.get(download_url, stream=True, timeout=(30, 180)) as response:
            response.raise_for_status()
            with partial_zip.open('wb') as destination:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        destination.write(chunk)
        partial_zip.replace(AIRESQ_ZIP)

    if AIRESQ_RAW.exists():
        shutil.rmtree(AIRESQ_RAW)
    AIRESQ_RAW.mkdir(parents=True)
    with zipfile.ZipFile(AIRESQ_ZIP) as archive:
        archive.extractall(AIRESQ_RAW)
    if not has_extracted_airesq(AIRESQ_RAW):
        raise FileNotFoundError(f'AIResQ archive extracted, but images/ and labels/ were not found under {AIRESQ_RAW}. Set AIRESQ_RAW to the extracted dataset root.')
    print('Extracted AIResQ to', AIRESQ_RAW)


## AIResQ annotations

AIResQ's existing person-only YOLO labels are valid for this training plan. Keep class `0` for person. Vehicle annotations are optional; the custom loss masks classification negatives at unmatched anchors on AIResQ samples, so an unlabeled vehicle is not taught as background. If you add reviewed vehicle boxes later, use class `1` and they will be used as positives. HIT-UAV provides full person and vehicle supervision.


In [ ]:
# Confirm the extracted source paths before proceeding.
image_dirs = [p for p in AIRESQ_RAW.rglob('*') if p.is_dir() and p.name.casefold() in {'images', 'image'}
              and any(f.is_file() and f.suffix.lower() in IMAGE_SUFFIXES for f in p.rglob('*'))]
label_dirs = [p for p in AIRESQ_RAW.rglob('*') if p.is_dir() and p.name.casefold() in {'labels', 'label'}
              and any(f.is_file() and f.suffix.lower() == '.txt' for f in p.rglob('*'))]
if len(image_dirs) != 1 or len(label_dirs) != 1:
    raise RuntimeError(f'Expected one AIResQ images/ and labels/ directory under {AIRESQ_RAW}; images={image_dirs}, labels={label_dirs}. Set AIRESQ_RAW to the correct extracted root.')
AIRESQ_IMAGES, AIRESQ_LABELS = image_dirs[0], label_dirs[0]
print('AIResQ images:', AIRESQ_IMAGES)
print('AIResQ labels:', AIRESQ_LABELS)


## Download HIT-UAV from Kaggle

Enter the Kaggle API token when prompted. The masked token is only held in the environment for the download command and is not saved to a file.


In [ ]:
from getpass import getpass
HIT_DOWNLOAD.mkdir(parents=True, exist_ok=True)
kaggle_token = getpass('Kaggle API token: ')
if not kaggle_token.strip():
    raise ValueError('A Kaggle API token is required.')
os.environ['KAGGLE_API_TOKEN'] = kaggle_token.strip()
try:
    subprocess.run(['kaggle', 'datasets', 'download', '-d',
                    'pandrii000/hituav-a-highaltitude-infrared-thermal-dataset',
                    '-p', str(HIT_DOWNLOAD), '--unzip'], check=True)
finally:
    os.environ.pop('KAGGLE_API_TOKEN', None)
print('HIT-UAV downloaded under', HIT_DOWNLOAD)


## Merge and remap both datasets

HIT-UAV keeps its official train/validation/test splits. AIResQ keeps supplied split folders or is split by capture groups from folder structure/metadata; a deterministic 80/10/10 image split is used only when groups are unavailable. Both datasets are copied into one YOLO dataset with `person: 0` and `vehicle: 1`; source prefixes prevent filename collisions.


In [ ]:
SPLITS = ('train', 'val', 'test')
CLASS_NAMES = ['person', 'vehicle']
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
ALIASES = {
    'person': 0, 'people': 0,
    'vehicle': 1, 'car': 1, 'bicycle': 1, 'bike': 1,
    'other vehicle': 1, 'othervehicle': 1, 'othervechicle': 1, 'other_vehicle': 1,
}
def norm_name(value):
    return ' '.join(str(value).casefold().replace('_', ' ').split())
def names_list(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x: int(x))]
    return list(names)
def find_yaml(root, required_splits=SPLITS):
    candidates = []
    for path in list(root.rglob('*.yaml')) + list(root.rglob('*.yml')):
        try: cfg = yaml.safe_load(path.read_text()) or {}
        except Exception: continue
        if isinstance(cfg, dict) and all(k in cfg for k in required_splits) and 'names' in cfg:
            candidates.append((path, cfg))
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one dataset YAML under {root}; found {[str(p) for p,_ in candidates]}')
    return candidates[0]
def split_dirs(root, yaml_path, cfg):
    root = Path(root).resolve()
    base = Path(cfg.get('path', '.'))
    if not base.is_absolute(): base = (yaml_path.parent / base).resolve()
    found = {}
    for split in SPLITS:
        value = Path(str(cfg[split]))
        options = [value] if value.is_absolute() else [base / value, yaml_path.parent / value, root / value]
        image_dir = next((p.resolve() for p in options if p.is_dir() and any(x.suffix.lower() in IMAGE_SUFFIXES for x in p.rglob('*'))), None)

        # Kaggle YAMLs can contain absolute paths from the dataset creator's machine
        # (for example /tmp/dataset/images/train). If that path is unavailable here,
        # resolve the split from the downloaded archive by its images/<split> layout.
        if image_dir is None:
            candidates = [p.resolve() for p in root.rglob(split)
                          if p.is_dir() and p.parent.name.casefold() == 'images'
                          and any(x.is_file() and x.suffix.lower() in IMAGE_SUFFIXES for x in p.rglob('*'))]
            with_labels = [p for p in candidates
                           if (p.parent.parent / 'labels' / split).is_dir()]
            if with_labels:
                candidates = with_labels
            if len(candidates) == 1:
                image_dir = candidates[0]
            else:
                raise FileNotFoundError(
                    f'Cannot resolve {split} image directory from YAML value {cfg[split]!r}. '
                    f'Searched the downloaded dataset at {root}; matching images/{split} folders: {candidates}'
                )
        found[split] = image_dir
    return found
def class_map(cfg, source_name):
    source_names = names_list(cfg['names'])
    mapping = {i: ALIASES[norm_name(name)] for i, name in enumerate(source_names) if norm_name(name) in ALIASES}
    ignored_ids = {i for i, name in enumerate(source_names)
                   if norm_name(name).replace(' ', '') == 'dontcare'}
    if set(mapping.values()) != {0, 1}:
        raise ValueError(f'{source_name} must contain person and vehicle classes. Found {source_names}')
    return mapping, ignored_ids
def write_converted_label(source_label, destination_label, id_map, source_name, ignored_ids=()):
    converted = []
    for line_number, line in enumerate(source_label.read_text().splitlines(), 1):
        if not line.strip(): continue
        fields = line.split()
        if len(fields) != 5: raise ValueError(f'Bad YOLO annotation at {source_label}:{line_number}: {line}')
        source_id = int(float(fields[0]))
        if source_id in ignored_ids:
            continue
        if source_id not in id_map:
            raise ValueError(f'Unknown class ID {source_id} in {source_label} for {source_name}.')
        fields[0] = str(id_map[source_id]); converted.append(' '.join(fields))
    destination_label.write_text(('\n'.join(converted) + '\n') if converted else '')
    return [int(line.split()[0]) for line in converted]
def copy_yolo_source(source_root, yaml_path, cfg, prefix, counts):
    image_dirs = split_dirs(source_root, yaml_path, cfg)
    id_map, ignored_ids = class_map(cfg, prefix)
    for split, image_dir in image_dirs.items():
        labels_dir = image_dir.parent.parent / 'labels' / split
        if not labels_dir.is_dir():
            matches = [p for p in source_root.rglob(split) if p.is_dir() and p.parent.name.casefold() == 'labels']
            if len(matches) != 1: raise FileNotFoundError(f'Cannot resolve labels/{split} for {prefix}')
            labels_dir = matches[0]
        dest_images = OUTPUT_DIR / split / 'images'; dest_labels = OUTPUT_DIR / split / 'labels'
        dest_images.mkdir(parents=True, exist_ok=True); dest_labels.mkdir(parents=True, exist_ok=True)
        for image_path in sorted(p for p in image_dir.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES):
            label_path = labels_dir / f'{image_path.stem}.txt'
            if not label_path.is_file(): raise FileNotFoundError(f'Missing label file for {image_path}')
            new_stem = f'{prefix}_{image_path.stem}'
            shutil.copy2(image_path, dest_images / f'{new_stem}{image_path.suffix.lower()}')
            ids = write_converted_label(label_path, dest_labels / f'{new_stem}.txt', id_map, prefix, ignored_ids)
            counts[split][prefix]['images'] += 1
            counts[split][prefix]['boxes'] += len(ids)
            counts[split][prefix]['person'] += ids.count(0)
            counts[split][prefix]['vehicle'] += ids.count(1)
def find_session_groups(dataset_root, images_dir, image_paths):
    groups = {}
    # Preserve session subfolders when the extracted images are grouped by capture.
    for image_path in image_paths:
        try:
            relative = image_path.relative_to(images_dir)
        except ValueError:
            relative = image_path.relative_to(dataset_root)
        if len(relative.parts) > 1:
            groups[image_path.name] = relative.parts[0]

    # Metadata schemas vary; accept common filename and session/sequence field names.
    metadata_files = list(dataset_root.rglob('metadata.json'))
    if len(metadata_files) == 1:
        metadata = json.loads(metadata_files[0].read_text())
        filename_keys = {'file', 'filename', 'file_name', 'image', 'image_name', 'image_path', 'path'}
        group_keys = {'session', 'session_id', 'sequence', 'sequence_id', 'flight', 'flight_id',
                      'capture', 'capture_id', 'location', 'site', 'mission', 'video_id'}
        known_names = {p.name for p in image_paths}
        def walk(node, parent_key=None):
            if isinstance(node, dict):
                lowered = {str(k).casefold(): v for k, v in node.items()}
                file_value = next((lowered[k] for k in filename_keys if k in lowered and isinstance(lowered[k], str)), None)
                group_value = next((lowered[k] for k in group_keys if k in lowered and isinstance(lowered[k], (str, int))), None)
                if group_value is not None:
                    candidates = [file_value, parent_key]
                    for candidate in candidates:
                        if isinstance(candidate, str) and Path(candidate).name in known_names:
                            groups[Path(candidate).name] = str(group_value)
                for key, value in node.items():
                    walk(value, str(key))
            elif isinstance(node, list):
                for item in node:
                    walk(item, parent_key)
        walk(metadata)
    return groups

def copy_flat_airesq(images_dir, labels_dir, counts):
    split_image_dirs = {split: images_dir / split for split in SPLITS}
    has_source_splits = all(
        folder.is_dir() and any(p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES for p in folder.rglob('*'))
        for folder in split_image_dirs.values()
    )
    if has_source_splits:
        image_paths = sorted(p for folder in split_image_dirs.values() for p in folder.rglob('*')
                             if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
        split_for = {p: next(split for split, folder in split_image_dirs.items() if folder in p.parents)
                     for p in image_paths}
        print('Preserving AIResQ train/val/test folders.')
    else:
        image_paths = sorted(p for p in images_dir.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
        split_for = None
    if not image_paths: raise FileNotFoundError(f'No AIResQ images found under {images_dir}')
    groups = find_session_groups(AIRESQ_RAW, images_dir, image_paths) if split_for is None else {}
    if split_for is None and len(set(groups.get(p.name) for p in image_paths if p.name in groups)) < 3:
        groups = {}
    if split_for is not None:
        pass
    elif groups and sum(p.name in groups for p in image_paths) >= 0.8 * len(image_paths):
        grouped_paths = {}
        for image_path in image_paths:
            grouped_paths.setdefault(groups.get(image_path.name, f'ungrouped:{image_path.name}'), []).append(image_path)
        shuffled_groups = list(grouped_paths.items())
        random.Random(42).shuffle(shuffled_groups)
        targets = {'train': 0.8 * len(image_paths), 'val': 0.1 * len(image_paths), 'test': 0.1 * len(image_paths)}
        assigned = {'train': 0, 'val': 0, 'test': 0}
        split_for = {}
        for group_name, members in shuffled_groups:
            split = max(('train', 'val', 'test'), key=lambda name: targets[name] - assigned[name])
            for member in members: split_for[member] = split
            assigned[split] += len(members)
        print('AIResQ split grouped by capture metadata/folders:', assigned)
    else:
        rng = random.Random(42)
        rng.shuffle(image_paths)
        n_train = int(len(image_paths) * 0.8)
        n_val = int(len(image_paths) * 0.1)
        split_for = {p: ('train' if i < n_train else 'val' if i < n_train + n_val else 'test')
                     for i, p in enumerate(image_paths)}
        print('AIResQ session groups were not identifiable; using deterministic image split.')
    id_map = {0: 0, 1: 1}  # AIResQ: 0=person; optional reviewed vehicle boxes use 1
    observed_ids = set()
    for image_path in image_paths:
        label_path = labels_dir / f'{image_path.stem}.txt'
        if not label_path.is_file():
            matches = list(labels_dir.rglob(f'{image_path.stem}.txt'))
            if len(matches) != 1: raise FileNotFoundError(f'Cannot uniquely locate label file for AIResQ image {image_path}; matches={matches}')
            label_path = matches[0]
        split = split_for[image_path]
        new_stem = f'airesq_{image_path.stem}'
        dest_images = OUTPUT_DIR / split / 'images'; dest_labels = OUTPUT_DIR / split / 'labels'
        dest_images.mkdir(parents=True, exist_ok=True); dest_labels.mkdir(parents=True, exist_ok=True)
        shutil.copy2(image_path, dest_images / f'{new_stem}{image_path.suffix.lower()}')
        ids = write_converted_label(label_path, dest_labels / f'{new_stem}.txt', id_map, 'AIResQ')
        observed_ids.update(ids)
        counts[split]['airesq']['images'] += 1
        counts[split]['airesq']['boxes'] += len(ids)
        counts[split]['airesq']['person'] += ids.count(0)
        counts[split]['airesq']['vehicle'] += ids.count(1)
    if 0 not in observed_ids:
        raise ValueError(f'AIResQ labels must contain at least one person box (class 0). Found class IDs {sorted(observed_ids)}.')
    print('AIResQ observed class IDs:', sorted(observed_ids), '(person-only labels are supported)')

if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
counts = {s: {'hit': {'images': 0, 'boxes': 0, 'person': 0, 'vehicle': 0},
              'airesq': {'images': 0, 'boxes': 0, 'person': 0, 'vehicle': 0}} for s in SPLITS}
hit_yaml, hit_cfg = find_yaml(HIT_DOWNLOAD)
copy_yolo_source(HIT_DOWNLOAD, hit_yaml, hit_cfg, 'hit', counts)
copy_flat_airesq(AIRESQ_IMAGES, AIRESQ_LABELS, counts)

for split in SPLITS:
    if not counts[split]['hit']['boxes'] + counts[split]['airesq']['boxes']:
        raise ValueError(f'{split} has no annotations')
    print(split, counts[split])
data_yaml = OUTPUT_DIR / 'person_vehicle.yaml'
data_yaml.write_text(
    'path: ' + json.dumps(str(OUTPUT_DIR)) + '\n' +
    'train: train/images\nval: val/images\ntest: test/images\n' +
    f'nc: 2\nnames: {json.dumps(CLASS_NAMES)}\n')
print('Wrote', data_yaml)


## Fine-tune the existing thermal checkpoint

Upload the current `uavbest.pt`. Its person and vehicle knowledge initializes training; HIT-UAV provides full two-class labels, while AIResQ adds high-resolution person examples. AIResQ vehicle boxes are not required. A partial-label criterion ignores classification loss on unmatched AIResQ anchors, preventing unlabeled objects from becoming background negatives. The old checkpoint is not modified.


In [ ]:
from google.colab import files
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils.loss import v8DetectionLoss
from ultralytics.utils.tal import make_anchors
from types import MethodType
import torch

class PartialLabelDetectionLoss(v8DetectionLoss):
    # Kept in sync with ultralytics==8.4.173. AIResQ files have an airesq_ prefix.
    def get_assigned_targets_and_loss(self, preds, batch):
        loss = torch.zeros(3, device=self.device)
        pred_distri, pred_scores = (preds['boxes'].permute(0, 2, 1).contiguous(),
                                    preds['scores'].permute(0, 2, 1).contiguous())
        anchor_points, stride_tensor = make_anchors(preds['feats'], self.stride, 0.5)
        dtype = pred_scores.dtype
        batch_size = pred_scores.shape[0]
        imgsz = torch.tensor(preds['feats'][0].shape[2:], device=self.device, dtype=dtype) * self.stride[0]
        targets = torch.cat((batch['batch_idx'].view(-1, 1), batch['cls'].view(-1, 1), batch['bboxes']), 1)
        targets = self.preprocess(targets.to(self.device), batch_size, scale_tensor=imgsz[[1, 0, 1, 0]])
        gt_labels, gt_bboxes = targets.split((1, 4), 2)
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0.0)
        pred_bboxes = self.bbox_decode(anchor_points, pred_distri)
        _, target_bboxes, target_scores, fg_mask, target_gt_idx = self.assigner(
            pred_scores.detach().sigmoid(), (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
            anchor_points * stride_tensor, gt_labels, gt_bboxes, mask_gt)
        target_scores_sum = target_scores.sum().clamp_(min=1)

        bce_loss = self.bce(pred_scores, target_scores.to(dtype))
        if self.class_weights is not None:
            bce_loss *= self.class_weights
        image_paths = batch.get('im_file')
        if image_paths is None or len(image_paths) != batch_size:
            raise RuntimeError('Partial-label training requires im_file paths in each batch.')
        partial_samples = torch.tensor(
            [Path(path).name.startswith('airesq_') for path in image_paths],
            device=self.device, dtype=torch.bool)
        # Keep positive matched-anchor classification supervision for labeled AIResQ objects.
        # Mask all unmatched-anchor classification terms only for AIResQ; HIT-UAV stays fully supervised.
        cls_mask = (~partial_samples[:, None] | fg_mask).unsqueeze(-1)
        bce_loss *= cls_mask.to(bce_loss.dtype)
        loss[1] = bce_loss.sum() / target_scores_sum
        loss[0], loss[2] = self.bbox_loss(
            pred_distri, pred_bboxes, anchor_points, target_bboxes / stride_tensor,
            target_scores, target_scores_sum, fg_mask, imgsz, stride_tensor)
        loss[0] *= self.hyp.box
        loss[1] *= self.hyp.cls
        loss[2] *= self.hyp.dfl
        return ((fg_mask, target_gt_idx, target_bboxes, anchor_points, stride_tensor), loss,
                dict(zip(self.loss_names, loss.detach())))

class PartialLabelDetectionTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        model.init_criterion = MethodType(lambda this: PartialLabelDetectionLoss(this), model)
        return model

uploaded = files.upload()
if not uploaded:
    raise ValueError('Upload the existing thermal checkpoint, for example uavbest.pt.')
checkpoint_name = next(iter(uploaded))
checkpoint_path = WORK_DIR / checkpoint_name
checkpoint_path.write_bytes(uploaded[checkpoint_name])
model = YOLO(str(checkpoint_path))
print('Starting checkpoint classes:', model.names)

model.train(data=str(data_yaml), trainer=PartialLabelDetectionTrainer, epochs=60, imgsz=1280, workers=2, batch=-1,
            project=str(WORK_DIR / 'runs'), name='airesq_hit_uav_person_vehicle',
            seed=42, patience=18, close_mosaic=10)

best_path = WORK_DIR / 'runs' / 'airesq_hit_uav_person_vehicle' / 'weights' / 'best.pt'
if not best_path.is_file(): raise FileNotFoundError(f'Training did not produce {best_path}')
trained_model = YOLO(str(best_path))
print('Trained classes:', trained_model.names)


## Evaluate and save the replacement candidate

AIResQ test images have person labels only, so report person AP there. HIT-UAV test images have both labels, so report person and vehicle AP there. Do not combine the two test sets into one metric: AIResQ has no vehicle ground truth, making combined vehicle metrics misleading. Compare the original checkpoint with the refined checkpoint on the same HIT-UAV test split when their class names are already `person`, `vehicle` in that order.


In [ ]:
def print_class_metrics(metrics, source):
    ap_by_class = {
        int(class_id): (float(metrics.box.ap50[row]), float(metrics.box.ap[row].mean()))
        for row, class_id in enumerate(metrics.box.ap_class_index)
    }
    class_ids = (0, 1) if source == 'hit' else (0,)
    for class_id in class_ids:
        label = CLASS_NAMES[class_id]
        if class_id not in ap_by_class:
            print(f'{source} {label}: no ground-truth boxes in this test split')
            continue
        ap50, ap = ap_by_class[class_id]
        print(f'{source} {label} AP50: {ap50:.4f}; AP50-95: {ap:.4f}')

for source in ('hit', 'airesq'):
    source_images = sorted((OUTPUT_DIR / 'test' / 'images').glob(f'{source}_*'))
    if not source_images:
        print(f'No {source} test images found; skipping source-specific metric.')
        continue
    source_test = WORK_DIR / f'{source}_only_test'
    if source_test.exists(): shutil.rmtree(source_test)
    source_test_images = source_test / 'images'; source_test_labels = source_test / 'labels'
    source_test_images.mkdir(parents=True); source_test_labels.mkdir(parents=True)
    for image_path in source_images:
        (source_test_images / image_path.name).symlink_to(image_path)
        label_path = OUTPUT_DIR / 'test' / 'labels' / f'{image_path.stem}.txt'
        (source_test_labels / label_path.name).symlink_to(label_path)
    source_yaml = WORK_DIR / f'{source}_test.yaml'
    source_yaml.write_text(
        'path: ' + json.dumps(str(source_test)) + '\n' +
        'train: images\nval: images\ntest: images\n' +
        f'nc: 2\nnames: {json.dumps(CLASS_NAMES)}\n')
    metrics = trained_model.val(data=str(source_yaml), split='test', imgsz=1280, plots=False,
                                project=str(WORK_DIR / 'source_eval'), name=source)
    print_class_metrics(metrics, source)
    if source == 'hit':
        baseline = YOLO(str(checkpoint_path))
        baseline_names = [str(value).casefold().replace('_', ' ').strip()
                          for value in (baseline.names.values() if isinstance(baseline.names, dict) else baseline.names)]
        if baseline_names == ['person', 'vehicle']:
            baseline_metrics = baseline.val(data=str(source_yaml), split='test', imgsz=1280, plots=False,
                                            project=str(WORK_DIR / 'source_eval'), name='hit_baseline')
            print('Original checkpoint on the same HIT-UAV test split:')
            print_class_metrics(baseline_metrics, 'hit')
        else:
            print('Baseline comparison skipped: checkpoint classes are not exactly [person, vehicle]:', baseline.names)

replacement_path = WORK_DIR / 'airesq_hit_uav_person_vehicle_best.pt'
shutil.copy2(best_path, replacement_path)
print('New two-class checkpoint:', replacement_path)
files.download(str(replacement_path))


## Before switching the app

Use the downloaded `airesq_hit_uav_person_vehicle_best.pt` as the replacement candidate after reviewing its predictions on held-out thermal scenes. Its labels are `person` and `vehicle`. Keep the existing `uavbest.pt` as a rollback copy; the notebook does not overwrite it.
